## 1.计算A股流通市值和A股总市值
## Calculate the free-float market capitalization and total market capitalization of A-shares

In [8]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

capital_dt = pd.read_parquet('data/capital.parquet')
capital_dt=capital_dt.sort_values(by=['stock_code', 'change_date'], ascending=[True, True])
capital_dt.rename(columns={'change_date': 'trade_date'}, inplace=True)
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
mv_factor_dt=pd.merge(price_dt, capital_dt, on=['stock_code', 'trade_date'], how='left')
mv_factor_dt.fillna(method='ffill', inplace=True)
mv_factor_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares
0,000001.SZ,20130104,15.99,355.0546,22.204789,443851.37,717567.5466,NaN,NaN,NaN,NaN
1,000001.SZ,20130107,16.30,361.9381,22.204789,357169.25,578450.4876,NaN,NaN,NaN,NaN
2,000001.SZ,20130108,16.00,355.2766,22.204789,312479.12,501360.0937,NaN,NaN,NaN,NaN
3,000001.SZ,20130109,15.86,352.1680,22.204789,251329.15,399696.1831,NaN,NaN,NaN,NaN
4,000001.SZ,20130110,15.87,352.3900,22.204789,240030.27,383347.7326,NaN,NaN,NaN,NaN


In [9]:
mv_factor_dt['float_a_shares_mv']=mv_factor_dt['float_a_shares']*mv_factor_dt['close_price']
mv_factor_dt['total_a_shares_mv']=mv_factor_dt['total_a_shares']*mv_factor_dt['close_price']
mv_factor_dt.tail()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv
12065642,920992.BJ,20251124,17.93,18.5230,1.033073,14658.45,26244.446,9674.163836,4740.627222,9673.609463,4739.70589,84982.926613,173447.817676
12065643,920992.BJ,20251125,18.05,18.6470,1.033073,10126.13,18280.210,9674.163836,4740.627222,9673.609463,4739.70589,85551.691320,174608.650812
12065644,920992.BJ,20251126,17.80,18.3887,1.033073,10567.74,19017.103,9674.163836,4740.627222,9673.609463,4739.70589,84366.764847,172190.248446
12065645,920992.BJ,20251127,17.65,18.2337,1.033073,7207.73,12856.675,9674.163836,4740.627222,9673.609463,4739.70589,83655.808964,170739.207026
12065646,920992.BJ,20251128,17.58,18.1614,1.033073,7501.27,13195.610,9674.163836,4740.627222,9673.609463,4739.70589,83324.029551,170062.054364


## 2.插入行业分类信息,计算市值因子
## Insert industry classification information and calculate market capitalization factors
*A股流通市值 = close_price * float_a_shares*

*A股总市值 = close_price * total_a_shares*

(PS:理论上小市值策略应该只看A股流通市值，直接跟资金盘相关)
(PS: The small-cap strategy should only consider the free-float market capitalization of A-shares, which is directly related to the money market.)

*市值因子1 = -Ln(A股流通市值) 全局标准化*
(Market capitalization factor 1: Global standardization of the free-float market capitalization of A-shares)

*市值因子2 = -Ln(A股总市值) 全局标准化*
(Market capitalization factor 2: Global standardization of the total market capitalization of A-shares)


In [10]:
cat_df=pd.read_parquet('data/ref_data/Stock_Industry_Year.parquet')
mv_factor_dt['industry_name'] = mv_factor_dt.assign(year=pd.to_datetime(mv_factor_dt['trade_date'].astype(str)).dt.year).merge(cat_df[['stock_code', 'year', 'industry_name']], on=['stock_code', 'year'], how='left')['industry_name']

In [11]:
import numpy as np
mv_factor_dt.dropna(inplace=True)
mv_factor_dt['mv_flt']=-np.log(mv_factor_dt['float_a_shares_mv'])
mv_factor_dt['mv_tot']=-np.log(mv_factor_dt['total_a_shares_mv'])
mv_factor_dt.head()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv,industry_name,mv_flt,mv_tot
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,9.765651e+06,银行,-15.593639,-16.094382
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,1.010893e+07,银行,-15.628187,-16.128930
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,9.167135e+06,银行,-15.530255,-16.031135


In [12]:
# 按交易日期分组，对当天全股票池的mv_flt列做Z-score标准化（(值-均值)/标准差）
mv_factor_dt['mv_flt_std'] = mv_factor_dt.groupby('trade_date')['mv_flt'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0  # 避免标准差为0时除以0报错
)

mv_factor_dt['mv_tot_std'] = mv_factor_dt.groupby('trade_date')['mv_tot'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0  # 避免标准差为0时除以0报错
)
mv_factor_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv,industry_name,mv_flt,mv_tot,mv_flt_std,mv_tot_std
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,9.765651e+06,银行,-15.593639,-16.094382,-1.744443,-2.218485
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781,-1.750973,-2.226354
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,1.010893e+07,银行,-15.628187,-16.128930,-1.774256,-2.250456
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781,-1.757097,-2.230350
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,9.167135e+06,银行,-15.530255,-16.031135,-1.726456,-2.194585


In [13]:
mv_factor_dt.tail()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv,industry_name,mv_flt,mv_tot,mv_flt_std,mv_tot_std
12065642,920992.BJ,20251124,17.93,18.5230,1.033073,14658.45,26244.446,9674.163836,4740.627222,9673.609463,4739.70589,84982.926613,173447.817676,医药生物,-11.350206,-12.063632,1.797748,1.432967
12065643,920992.BJ,20251125,18.05,18.6470,1.033073,10126.13,18280.210,9674.163836,4740.627222,9673.609463,4739.70589,85551.691320,174608.650812,医药生物,-11.356876,-12.070302,1.805374,1.440739
12065644,920992.BJ,20251126,17.80,18.3887,1.033073,10567.74,19017.103,9674.163836,4740.627222,9673.609463,4739.70589,84366.764847,172190.248446,医药生物,-11.342929,-12.056355,1.809709,1.446056
12065645,920992.BJ,20251127,17.65,18.2337,1.033073,7207.73,12856.675,9674.163836,4740.627222,9673.609463,4739.70589,83655.808964,170739.207026,医药生物,-11.334466,-12.047893,1.821782,1.458540
12065646,920992.BJ,20251128,17.58,18.1614,1.033073,7501.27,13195.610,9674.163836,4740.627222,9673.609463,4739.70589,83324.029551,170062.054364,医药生物,-11.330492,-12.043919,1.837061,1.474868


## 3.初步回测：覆盖全行业指定数量选股
## Backtest: Cover all industries with a specified number of stocks
遍历每个日期，从中证1000基金持仓中遴选小盘股，可以基本剔除了严重问题股、流动性枯竭股，收益温和，回撤更小。
以mv_flt_std单个因子为例，所有样本在当日排序后winsorize，取1%分位数作为下限，99%分位数作为上限。然后为了分散风险，每个行业选stock_num只股票生成一个选股的df
Iterate through each date, select small-cap stocks from the CSI 1000 holdings, which can basically eliminate severely problematic stocks and illiquid stocks, achieving mild returns and smaller drawdowns.
Take the single factor mv_flt_std as an example: After winsorization, select stock_num stocks per industry to generate a stock selection DataFrame.

In [14]:
from scipy.stats.mstats import winsorize  # 专业的缩尾处理函数

stock_num = 10
# 读取数据
fund_df = pd.read_parquet('data/ref_data/ETF_hold_512100.SH.parquet')

# ===================== 1. 数据预处理 =====================
# 处理mv_factor_dt的日期格式（数字转日期）
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'].astype(str))

# 处理基金持仓数据的日期格式
fund_df = fund_df.rename(columns={'end_date': 'report_end_date'})  # 重命名避免混淆
fund_df['report_end_date'] = pd.to_datetime(fund_df['report_end_date'])

# 生成基金持仓的有效时间区间（半年报：1-6月，年报：7-12月）
def get_position_valid_period(row):
    """计算基金持仓的有效时间区间"""
    year = row['report_year']
    if row['report_type'] == '中报':
        start_date = pd.to_datetime(f'{year}-01-01')
        end_date = pd.to_datetime(f'{year}-06-30')
    elif row['report_type'] == '年报':
        start_date = pd.to_datetime(f'{year}-07-01')
        end_date = pd.to_datetime(f'{year}-12-31')
    else:
        start_date = end_date = row['report_end_date']
    return pd.Series([start_date, end_date], index=['pos_start_date', 'pos_end_date'])

# 给基金持仓添加有效时间区间
fund_df[['pos_start_date', 'pos_end_date']] = fund_df.apply(get_position_valid_period, axis=1)

# 提取中证1000基金持仓的股票代码+有效区间（去重）
fund_holdings = fund_df[['stock_code', 'pos_start_date', 'pos_end_date']].drop_duplicates()

# ===================== 2. 核心选股逻辑 =====================
def select_stocks_by_date(date_group):
    """
    对单个日期的样本执行选股：
    1. 筛选当日属于中证1000持仓的股票
    2. 对mv_flt_std列做1%/99%缩尾
    3. 按行业分组，每个独特行业选stock_num只（该行业内缩尾后MV最高的）
    """
    # 获取当日日期和数据
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 步骤1：筛选当日属于中证1000基金持仓的股票
    valid_holdings = fund_holdings[
        (fund_holdings['pos_start_date'] <= trade_date) & 
        (fund_holdings['pos_end_date'] >= trade_date)
    ]['stock_code'].tolist()
    holding_stocks = daily_data[daily_data['stock_code'].isin(valid_holdings)]
    
    if len(holding_stocks) == 0:
        return pd.DataFrame()  # 无持仓股票则返回空
    
    # 步骤2：对mv_flt_std列做1%（下限）、99%（上限）的缩尾处理
    holding_stocks['mv_flt_std'] = winsorize(
        holding_stocks['mv_flt_std'].values, 
        limits=(0.01, 0.01),
        inclusive=(True, True)
    )
    
    # 步骤3：按行业分组，每个行业选缩尾后MV最高的stock_num只
    # 过滤掉行业名称为空/NaN的记录（避免无意义分组）
    holding_stocks = holding_stocks.dropna(subset=['industry_name'])
    # 按行业分组，每组取mv_flt_std最大的stock_num只
    selected = holding_stocks.groupby('industry_name', group_keys=False).apply(
        lambda x: x.sort_values('mv_flt_std', ascending=False).head(stock_num)
    )
    
    # 添加选股标识：按缩尾后MV降序排名
    selected = selected.sort_values('mv_flt_std', ascending=False)
    selected['selection_rank'] = range(1, len(selected)+1)  # 排名按整体MV从高到低
    
    return selected

# 按交易日分组执行选股
selected_stocks = mv_factor_dt.groupby('trade_date').apply(select_stocks_by_date)

# 重置索引
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列
final_selection_df = selected_stocks[['trade_date', 'stock_code', 'industry_name', 'mv_flt', 'mv_flt_std', 'selection_rank']]
final_selection_df.tail()

,trade_date,stock_code,industry_name,mv_flt,mv_flt_std,selection_rank
626247,2025-06-30,601375.SH,非银金融,-14.176229,-0.796226,290.0
626248,2025-06-30,002948.SZ,银行,-14.253634,-0.864398,291.0
626249,2025-06-30,600621.SH,非银金融,-14.300680,-0.905832,292.0
626250,2025-06-30,603983.SH,美容护理,-14.339889,-0.940364,293.0
626251,2025-06-30,000686.SZ,非银金融,-14.395309,-0.989173,294.0


In [15]:
# 初步回测选股文件
final_selection_df.to_parquet('records/mv_selection_v1.parquet', index=False)

In [16]:
import pandas as pd
final_selection_df=pd.read_parquet('records/mv_selection_v1.parquet')
final_selection_df.head()

,trade_date,stock_code,industry_name,mv_flt,mv_flt_std,selection_rank
0,2016-07-01,002346.SZ,建筑材料,-11.933907,1.396203,1.0
1,2016-07-01,603299.SH,基础化工,-11.952764,1.396203,2.0
2,2016-07-01,002726.SZ,食品饮料,-11.987691,1.396203,3.0
3,2016-07-01,603800.SH,机械设备,-11.978876,1.396203,4.0
4,2016-07-01,603166.SH,汽车,-11.814988,1.396203,5.0


In [17]:
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
price_dt.head()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover
2983,000001.SZ,20130104,15.99,355.0546,22.204789,443851.37,717567.5466
8347,000001.SZ,20130107,16.30,361.9381,22.204789,357169.25,578450.4876
10530,000001.SZ,20130108,16.00,355.2766,22.204789,312479.12,501360.0937
13553,000001.SZ,20130109,15.86,352.1680,22.204789,251329.15,399696.1831
10100,000001.SZ,20130110,15.87,352.3900,22.204789,240030.27,383347.7326


基金持仓的年报数据最早从2016年7月1日开始。鉴于此，回测区间为2016年7月1日到2025年6月30日
交易日收盘时选股，需要次交易日以复权收盘价执行买入。每5个交易日以复权收盘价卖出调整一次持仓，收盘时按照复权收盘价买入，进行下一个循环
为了回测简便，交易无手续费。如果要加，则要结合调仓千一手续费。
The annual report data of fund positions starts from July 1, 2016. In view of this, the backtest period is set from July 1, 2016 to June 30, 2025.
Stock selection is conducted at the close of each trading day, and purchases are executed at the adjusted closing price on the next trading day. Positions are sold and rebalanced at the adjusted closing price every 5 trading days, followed by purchasing at the adjusted closing price.
For simplicity of backtesting, **no transaction fees are applied.** If fees are to be included, we'll add a 0.1% commission.(过户费、佣金、印花税)

In [18]:
import pandas as pd
import numpy as np
import warnings
from util import *

warnings.filterwarnings("ignore")

# ----------------------------
# User‑configurable parameters
# ----------------------------
INITIAL_CASH = 1_000_000        # initial capital
START_DATE = '2016-07-01'       # backtest start date (inclusive)
END_DATE   = '2025-06-30'       # backtest end date (inclusive)
REBALANCE_FREQ = 5             # rebalance every N calendar days (using available selection dates)
TOP_N = 5                       # number of top‑ranked stocks to hold each rebalance

PRICE_FILE = 'data/eod_prices.parquet'   # path to price file
SELECTION_FILE = 'records/mv_selection_v1.parquet'  # path to selection file

# ----------------------------
# Load and prepare data
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)
selection_df = load_selection(SELECTION_FILE)

# Create pivot table: rows = dates, columns = stocks, values = adjusted_close
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
# Ensure index is datetime (fix for categorical index)
price_pivot.index = pd.to_datetime(price_pivot.index)

# Filter by date range
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# Get sorted unique selection dates (these are the only dates we consider for rebalancing)
selection_dates = sorted(selection_df['trade_date'].unique())
if not selection_dates:
    raise ValueError("No selection data available in the given date range.")

# Determine rebalance dates (every REBALANCE_FREQ-th date in the selection_dates list)
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# Generate the full timeline (all trading days in the pivot index)
all_dates = price_pivot.index.tolist()
# Ensure we start from the first rebalance date (before that we have no positions)
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# ----------------------------
# Backtest loop
# ----------------------------
cash = INITIAL_CASH
holdings = {}  # stock_code -> shares
portfolio_values = []
benchmark_cumulative = None

# Pre‑compute benchmark returns (daily equal‑weighted average)
benchmark_daily, benchmark_cum = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)

# For each day in timeline
for date in timeline:
    # Get today's prices for all stocks (if missing, price will be NaN)
    today_prices = price_pivot.loc[date]

    # --- Rebalance if today is a rebalance date ---
    if date in rebalance_dates:
        # 1. Sell all current holdings
        for stock, shares in holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash += shares * price
            # If price is missing, we assume the stock is worthless (sell for 0)
        holdings = {}

        # 2. Get selected stocks for today
        selected = get_top_n_selection(selection_df, date, TOP_N)

        # 3. Buy new holdings equally
        if selected:
            # Filter stocks that have a valid price today
            valid_stocks = [s for s in selected if pd.notna(today_prices.get(s, np.nan))]
            if valid_stocks:
                amount_per_stock = cash / len(valid_stocks)
                for stock in valid_stocks:
                    price = today_prices[stock]
                    shares = amount_per_stock / price
                    holdings[stock] = shares
                cash = 0.0  # all cash invested
            else:
                # No valid stocks – keep cash
                pass

    # --- Calculate total portfolio value today ---
    portfolio_value = cash
    for stock, shares in holdings.items():
        price = today_prices.get(stock, np.nan)
        if pd.notna(price):
            portfolio_value += shares * price
        # If price is missing, the position contributes 0
    portfolio_values.append(portfolio_value)

# ----------------------------
# Prepare results
# ----------------------------
portfolio_series = pd.Series(portfolio_values, index=timeline, name='Portfolio')

benchmark_series = benchmark_cum.reindex(timeline, method='ffill').fillna(1.0)
benchmark_funds = INITIAL_CASH * benchmark_series

portfolio_returns = portfolio_series.pct_change().dropna()

metrics = compute_metrics(portfolio_returns, rf=0, periods_per_year=252)
print("\n--- Performance Metrics ---")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

plot_results(portfolio_series, benchmark_funds, timeline,
             title='Backtest: Top {} Stocks Rebalanced Every {} Days'.format(TOP_N, REBALANCE_FREQ))

Loading data...
Number of rebalance dates: 437

--- Performance Metrics ---
Annualized Return: 0.0797
Volatility: 0.3203
Sharpe Ratio: 0.2487
Max Drawdown: -0.5384


## 4.项目方案：分位数回测(Long-Only/Long-Short)
## Backtest: Quantile selection (10/90)
Monthly rebalance 4 times


In [19]:
from scipy.stats.mstats import winsorize

# 读取数据）
fund_df = pd.read_parquet('data/ref_data/ETF_hold_512100.SH.parquet')

# ===================== 1. 数据预处理 =====================
# 处理mv_factor_dt的日期格式（数字转日期）
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'].astype(str))

# 处理基金持仓数据的日期格式
fund_df = fund_df.rename(columns={'end_date': 'report_end_date'})
fund_df['report_end_date'] = pd.to_datetime(fund_df['report_end_date'])

# 生成基金持仓的有效时间区间（半年报：1-6月，年报：7-12月）
def get_position_valid_period(row):
    year = row['report_year']
    if row['report_type'] == '中报':
        start_date = pd.to_datetime(f'{year}-01-01')
        end_date = pd.to_datetime(f'{year}-06-30')
    elif row['report_type'] == '年报':
        start_date = pd.to_datetime(f'{year}-07-01')
        end_date = pd.to_datetime(f'{year}-12-31')
    else:
        start_date = end_date = row['report_end_date']
    return pd.Series([start_date, end_date], index=['pos_start_date', 'pos_end_date'])

fund_df[['pos_start_date', 'pos_end_date']] = fund_df.apply(get_position_valid_period, axis=1)

# 提取中证1000基金持仓的股票代码+有效区间（去重）
fund_holdings = fund_df[['stock_code', 'pos_start_date', 'pos_end_date']].drop_duplicates()

# ===================== 2. 核心选股逻辑（分位数多空） =====================
def select_stocks_by_date(date_group):
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 筛选当日属于中证1000基金持仓的股票
    valid_holdings = fund_holdings[
        (fund_holdings['pos_start_date'] <= trade_date) &
        (fund_holdings['pos_end_date'] >= trade_date)
    ]['stock_code'].tolist()
    holding_stocks = daily_data[daily_data['stock_code'].isin(valid_holdings)]
    
    if len(holding_stocks) == 0:
        return pd.DataFrame()
    
    # 对mv_flt_std列做1%/99%缩尾
    holding_stocks['mv_flt_std'] = winsorize(
        holding_stocks['mv_flt_std'].values, 
        limits=(0.01, 0.01),
        inclusive=(True, True)
    )
    
    # 计算分位数并标记多空
    q10 = holding_stocks['mv_flt_std'].quantile(0.10)
    q90 = holding_stocks['mv_flt_std'].quantile(0.90)
    
    holding_stocks['signal'] = 'none'
    holding_stocks.loc[holding_stocks['mv_flt_std'] <= q10, 'signal'] = 'long'
    holding_stocks.loc[holding_stocks['mv_flt_std'] >= q90, 'signal'] = 'short'
    
    return holding_stocks

# 按交易日分组执行选股
selected_stocks = mv_factor_dt.groupby('trade_date').apply(select_stocks_by_date)

# 重置索引
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列
final_selection_df = selected_stocks[['trade_date', 'stock_code', 'industry_name', 
                                       'mv_flt_std', 'signal']]
final_selection_df.to_parquet('records/mv_selection_v2.parquet', index=False)


In [22]:
final_selection_df.head(10)

,trade_date,stock_code,industry_name,mv_flt_std,signal
0,2016-07-01,000005.SZ,房地产,0.081375,none
1,2016-07-01,000010.SZ,建筑装饰,0.891117,short
2,2016-07-01,000011.SZ,房地产,1.211196,short
3,2016-07-01,000014.SZ,房地产,-1.655680,long
4,2016-07-01,000016.SZ,家用电器,-0.025441,none
5,2016-07-01,000018.SZ,房地产,0.496187,none
6,2016-07-01,000034.SZ,计算机,-0.021068,none
7,2016-07-01,000035.SZ,环保,0.290198,none
8,2016-07-01,000036.SZ,房地产,-0.224212,none
9,2016-07-01,000038.SZ,房地产,1.188688,short


In [3]:
import pandas as pd
import numpy as np
import warnings
from util import *

warnings.filterwarnings("ignore")

# ----------------------------
# User‑configurable parameters
# ----------------------------
INITIAL_CASH = 1_000_000
START_DATE = '2016-07-01'
END_DATE   = '2025-06-30'
REBALANCE_FREQ = 5

PRICE_FILE = 'data/eod_prices.parquet'
SELECTION_FILE = 'records/mv_selection_v2.parquet'
TRADE_LOG_FILE = 'records/mv_trades.csv'   # where to save trade records for long‑only

# ----------------------------
# Load and prepare data
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)

# Load selection file directly
selection_df = pd.read_parquet(SELECTION_FILE)
selection_df = selection_df[['trade_date', 'stock_code', 'signal']].copy()
selection_df['trade_date'] = pd.to_datetime(selection_df['trade_date'])

# Price pivot
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
price_pivot.index = pd.to_datetime(price_pivot.index)

# Filter by date range
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# Rebalance dates
selection_dates = sorted(selection_df['trade_date'].unique())
if not selection_dates:
    raise ValueError("No selection data available.")
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

all_dates = price_pivot.index.tolist()
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# Benchmark
benchmark_daily, benchmark_cum = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)

# ----------------------------
# Helper to get all stocks by signal
# ----------------------------
def get_stocks_by_signal(date, signal_type):
    day_data = selection_df[selection_df['trade_date'] == date]
    day_data = day_data[day_data['signal'] == signal_type]
    return day_data['stock_code'].tolist()

# ----------------------------
# Backtest function with optional trade recording
# ----------------------------
def run_backtest(mode='long_only', record_trades=False):
    cash = INITIAL_CASH
    holdings = {}
    portfolio_values = []
    trades = []   # list of trade dicts if record_trades is True

    for date in timeline:
        today_prices = price_pivot.loc[date]

        if date in rebalance_dates:
            # --- Close existing positions (sell) ---
            for stock, shares in holdings.items():
                price = today_prices.get(stock, np.nan)
                if pd.notna(price):
                    cash += shares * price
                    if record_trades:
                        trades.append({
                            'trade_date': date,
                            'stock_code': stock,
                            'side': 'sell',
                            'shares': shares,
                            'price': price
                        })
            holdings = {}

            # --- Build new portfolio based on signals ---
            if mode == 'long_only':
                long_stocks = get_stocks_by_signal(date, 'long')
                valid = [s for s in long_stocks if pd.notna(today_prices.get(s, np.nan))]
                if valid:
                    amount_per_stock = cash / len(valid)
                    for stock in valid:
                        price = today_prices[stock]
                        shares = amount_per_stock / price
                        holdings[stock] = shares
                        if record_trades:
                            trades.append({
                                'trade_date': date,
                                'stock_code': stock,
                                'side': 'buy',
                                'shares': shares,
                                'price': price
                            })
                    cash = 0.0

            else:  # long_short
                long_stocks = get_stocks_by_signal(date, 'long')
                short_stocks = get_stocks_by_signal(date, 'short')
                valid_long = [s for s in long_stocks if pd.notna(today_prices.get(s, np.nan))]
                valid_short = [s for s in short_stocks if pd.notna(today_prices.get(s, np.nan))]
                n_long = len(valid_long)
                n_short = len(valid_short)

                if n_long == 0 and n_short == 0:
                    continue

                if n_long > 0 and n_short > 0:
                    cash_per_side = cash / 2
                    # Longs
                    amount_per_long = cash_per_side / n_long
                    for stock in valid_long:
                        price = today_prices[stock]
                        shares = amount_per_long / price
                        holdings[stock] = shares
                        if record_trades:
                            trades.append({
                                'trade_date': date,
                                'stock_code': stock,
                                'side': 'buy',
                                'shares': shares,
                                'price': price
                            })
                    # Shorts
                    amount_per_short = cash_per_side / n_short
                    for stock in valid_short:
                        price = today_prices[stock]
                        shares = -amount_per_short / price
                        holdings[stock] = shares
                        if record_trades:
                            trades.append({
                                'trade_date': date,
                                'stock_code': stock,
                                'side': 'sell_short',
                                'shares': -shares,  # positive shares for record
                                'price': price
                            })
                elif n_long > 0:
                    amount_per_stock = cash / n_long
                    for stock in valid_long:
                        price = today_prices[stock]
                        shares = amount_per_stock / price
                        holdings[stock] = shares
                        if record_trades:
                            trades.append({
                                'trade_date': date,
                                'stock_code': stock,
                                'side': 'buy',
                                'shares': shares,
                                'price': price
                            })
                    cash = 0.0
                else:  # only shorts
                    amount_per_stock = cash / n_short
                    for stock in valid_short:
                        price = today_prices[stock]
                        shares = -amount_per_stock / price
                        holdings[stock] = shares
                        if record_trades:
                            trades.append({
                                'trade_date': date,
                                'stock_code': stock,
                                'side': 'sell_short',
                                'shares': -shares,
                                'price': price
                            })
                    cash += cash

        # Daily valuation
        portfolio_value = cash
        for stock, shares in holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                portfolio_value += shares * price
        portfolio_values.append(portfolio_value)

    # Build results
    portfolio_series = pd.Series(portfolio_values, index=timeline, name='Portfolio')
    benchmark_series = benchmark_cum.reindex(timeline, method='ffill').fillna(1.0)
    benchmark_funds = INITIAL_CASH * benchmark_series
    portfolio_returns = portfolio_series.pct_change().dropna()
    metrics = compute_metrics(portfolio_returns, rf=0, periods_per_year=252)

    if record_trades:
        trades_df = pd.DataFrame(trades)
        return portfolio_series, benchmark_funds, metrics, trades_df
    else:
        return portfolio_series, benchmark_funds, metrics

# ----------------------------
# Run long‑only with trade recording
# ----------------------------
print("\n" + "="*50)
print("Running Long‑Only Backtest (all long signals) - with trade recording")
print("="*50)
port_long, bench_long, metrics_long, trades_df = run_backtest(mode='long_only', record_trades=True)

# Save trades to CSV
if not trades_df.empty:
    trades_df.to_csv(TRADE_LOG_FILE, index=False)
    print(f"Trade records saved to {TRADE_LOG_FILE}")
else:
    print("No trades recorded.")

print("\n--- Performance Metrics (Long‑Only) ---")
for k, v in metrics_long.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# ----------------------------
# Run long‑short (no trade recording)
# ----------------------------
print("\n" + "="*50)
print("Running Long‑Short Backtest (all long and short signals)")
print("="*50)
port_ls, bench_ls, metrics_ls = run_backtest(mode='long_short', record_trades=False)
print("\n--- Performance Metrics (Long‑Short) ---")
for k, v in metrics_ls.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# plotting
plot_results(port_long, bench_long, timeline, title='Long‑Only (all long signals)', plot_excess=True)
plot_results(port_ls, bench_ls, timeline, title='Long‑Short (all signals)', plot_excess=True)

Loading data...
Number of rebalance dates: 437

Running Long‑Only Backtest (all long signals) - with trade recording
Trade records saved to records/mv_trades.csv

--- Performance Metrics (Long‑Only) ---
Annualized Return: -0.0663
Volatility: 0.2396
Sharpe Ratio: -0.2767
Max Drawdown: -0.5717

Running Long‑Short Backtest (all long and short signals)

--- Performance Metrics (Long‑Short) ---
Annualized Return: -0.0428
Volatility: 0.0733
Sharpe Ratio: -0.5841
Max Drawdown: -0.3803


分位数策略失效的原因？
平均自由流通市值近百亿元；远高于典型小市值策略标的 (通常 < 20 亿元)
Reasons for the failure of the quantile strategy
The average free-float market capitalization is close to 10 billion yuan for CSI1000 holding, far higher than the typical underlying assets for small-cap strategies (usually less than 2 billion yuan).